In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
N = 60  # пользователей в каждой группе

def make_group(label, n, p_conv, time_mu, time_sd,
               clicks_lam, sessions_lam, conv_bonus_time):
    converted = np.random.binomial(1, p_conv, n)

    # Выручка: только у конвертировавших (логичнормальное распределение → правый хвост)
    revenue = np.where(
        converted == 1,
        np.random.lognormal(mean=5.4, sigma=0.7, size=n).round(2),
        0.0
    )

    # Время на странице: нормальное + бонус конвертировавшим
    time_sec = np.random.normal(time_mu, time_sd, n) + converted * conv_bonus_time
    time_sec = np.clip(time_sec, 5, None).round(1)

    # Счётные метрики
    clicks = np.random.poisson(clicks_lam, n)
    sessions = np.random.poisson(sessions_lam, n) + 1

    # Категориальная
    device = np.random.choice(["mobile", "desktop", "tablet"],
                              size=n, p=[0.55, 0.35, 0.10])

    return pd.DataFrame({
        "user_id":  [f"{label}_{i:03d}" for i in range(n)],
        "group":    label,
        "converted": converted,
        "revenue":  revenue,
        "time_sec": time_sec,
        "clicks":   clicks,
        "sessions": sessions,
        "device":   device,
    })

df = pd.concat([
    make_group("A", N, p_conv=0.10, time_mu=45, time_sd=15,
               clicks_lam=3, sessions_lam=1.5, conv_bonus_time=20),
    make_group("B", N, p_conv=0.20, time_mu=52, time_sd=18,
               clicks_lam=4, sessions_lam=1.6, conv_bonus_time=25),
], ignore_index=True)

df.to_csv("ab_data.csv", index=False)
print(df.head(10).to_string(index=False))
print("shape:", df.shape)

user_id group  converted  revenue  time_sec  clicks  sessions  device
  A_000     A          0     0.00      42.6       4         2  mobile
  A_001     A          1   133.77      71.1       2         2  mobile
  A_002     A          0     0.00      73.3       6         1  mobile
  A_003     A          0     0.00      47.6       3         1  mobile
  A_004     A          0     0.00      48.9       0         5 desktop
  A_005     A          0     0.00      43.9       5         3  mobile
  A_006     A          0     0.00      16.2       0         2 desktop
  A_007     A          0     0.00      44.6       6         2 desktop
  A_008     A          0     0.00      45.9       2         4  mobile
  A_009     A          0     0.00      81.9       4         4 desktop
shape: (120, 8)
